In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
local_path = r"D:\Gemma4 2B 4B\gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(local_path)
tokenizer = processor.tokenizer
model = AutoModelForImageTextToText.from_pretrained(
    local_path,
    low_cpu_mem_usage=True,  
)

model.to('cpu') 

def generate(prompt_text, tokenizer, max_new_tokens= 1024):

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id= model.tokenizer.eos_token_id 
    )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    
    return model.tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
db = lancedb.connect(r'D:\LLMOps\lance_db\Attention_paper_db')
transformers_table = db.open_table('transformer_table') 

all_rows = transformers_table.to_pandas() 
all_rows.head()

,id,client_id,doc_id,source_file,chunk_index,text,vector
0,Attention_is_all_you_need_chunk_0000,Advanced_RAG_Method,Attention_is_all_you_need,Attention is that all you need.pdf,0,3 2023 2 0 2 g u A 2 ] L C . s c [ 7 v 2 6 7 3...,"[0.020600747, -0.0024948071, -0.022079397, -0...."
1,Attention_is_all_you_need_chunk_0001,Advanced_RAG_Method,Attention_is_all_you_need,Attention is that all you need.pdf,1,Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com...,"[0.023133932, -0.022282515, -0.009542208, -0.0..."
2,Attention_is_all_you_need_chunk_0002,Advanced_RAG_Method,Attention_is_all_you_need,Attention is that all you need.pdf,2,"1 Introduction Recurrent neural networks, long...","[0.021390695, -0.0020416682, 0.004080007, -0.0..."
3,Attention_is_all_you_need_chunk_0003,Advanced_RAG_Method,Attention_is_all_you_need,Attention is that all you need.pdf,3,2 Background The goal of reducing sequential c...,"[0.0036105474, 0.002861142, -0.016650109, -0.0..."
4,Attention_is_all_you_need_chunk_0004,Advanced_RAG_Method,Attention_is_all_you_need,Attention is that all you need.pdf,4,"Here is a comprehensive, searchable descriptio...","[0.021273963, -0.012828157, -0.012067666, -0.0..."


In [2]:
import lancedb 
from langchain_core.documents import Document

import sys
sys.path.append(r'D:\LLMOps\pyfiles')
from EmbedModelLoader import EmbeddingManager #function
from langchain_classic.retrievers import BM25Retriever 
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.retrievers import BaseRetriever
from typing import List
from pydantic import Field 
from config_loader import load_config  

config = load_config()

db = lancedb.connect(r'D:\LLMOps\lance_db\Attention_paper_db')
transformers_table = db.open_table('transformer_table') 

all_rows = transformers_table.to_pandas()

# Rebuild LangChain Document objects from the stored rows
bm25_documents = [
    Document(
        page_content=row['text'],
        metadata={
        }
    )
    for _, row in all_rows.iterrows()
]
 
# Rebuild LangChain Document objects from the stored rows
bm25_documents = [
    Document(
        page_content=row['text'],
        metadata={
            'id': row['id'],
            'chunk_index': row['chunk_index'],
            'source_file': row['source_file'],
            'client_id': row['client_id'],
        }
    ) 
    for _, row in all_rows.iterrows()
] 

class LanceDBDirectRetriever(BaseRetriever):  
    table: object = Field(...)
    embedding_manager: object = Field(...)
    top_k: int = 1

    def _get_relevant_documents(self, query: str) -> List[Document]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        results = (
            self.table.search(query_embedding, vector_column_name="vector")
            .nprobes(2)
            .limit(self.top_k)
            .to_pandas()
        )

        return [
            Document(
                page_content=row['text'],
                metadata={
                    'id': row['id'],
                    'chunk_index': row['chunk_index'],
                    'source_file': row['source_file'], 
                }
            )
            for _, row in results.iterrows()
        ] 
    

def Hybrid_retriever(query, table, embedding_manager, top_k = 5): 
    top_k= config['retriever']['top_k'] 

    vector_retriever = LanceDBDirectRetriever(
        table=transformers_table,
        embedding_manager=embedding_manager,
        top_k=top_k 
    )
   
    bm25_retriever = BM25Retriever.from_documents(bm25_documents)
    bm25_retriever.k = top_k 

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.5, 0.5]
    )
    
    results = ensemble_retriever.invoke(query)
    return results

In [7]:
import sys
sys.path.append(r'D:\LLMOps\pyfiles')   
from query_variationar import query_variations   
from HybridSearch import Hybrid_search   

query = 'Recurrent models typically factor computation along the symbol positions of'

queries = query_variations(query, 0.80) 
queries.append(query)

Batches: 100%|██████████| 1/1 [00:23<00:00, 23.61s/it]


In [4]:
chunks=  []

for query in queries:
    res = Hybrid_search(query, transformers_table, embedding_manager, top_k = 5) 
    chunks.extend(res)  


NameError: name 'queries' is not defined

In [6]:
queries

['How do recurrent models factor computation based on symbol positions?',
 'What is the factorization of computation in recurrent models along symbol positions?',
 'Recurrent models typically factor computation along the symbol positions of']

In [ ]:
chunks  

[Document(metadata={'id': 'Attention_is_all_you_need_chunk_0002', 'chunk_index': 2, 'source_file': 'Attention is that all you need.pdf'}, page_content='1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15]. Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence len

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
#`pip install einops 
LOCAL_PATH = r"D:\LLMOps\RAG Learn\Models\jina-reranker-v2-base-multilingual"

model = HuggingFaceCrossEncoder(  
    model_name=LOCAL_PATH, 
    model_kwargs={"trust_remote_code": True, 
    "model_kwargs": {"dtype": "auto"}
    },
  )
reranker = CrossEncoderReranker(model=model, top_n=5)   

C:\Users\HP\AppData\Local\Temp\ipykernel_1544\1366271878.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cross_encoders import HuggingFaceCrossEncoder
d:\LLMOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] You are using a model of type `xlm-roberta` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


In [3]:
reranked_docs = reranker.compress_documents(chunks, query= query) 

NameError: name 'chunks' is not defined

In [ ]:
import sys 
sys.path.append(r'D:\LLMOps\pyfiles')
from RetrievalSystem import RAGRetriever  

import lancedb 
db = lancedb.connect(r'D:\LLMOps\lance_db\Attention_paper_db')
transformers_table = db.open_table('transformer_table') 

ok


# RAG Raranker: 
 
 A reranker is a specoalized AI model that acts as 'quality inspector' that improves the initial search result by reordering them based on their semantic meaning to the query.

 embedding of Query and Search results.

 





In [62]:
import sys
sys.path.append(r'D:\LLMOps\pyfiles')
from HybridSearch import Hybrid_search 

res = Hybrid_search(query, transformers_table, embedding_manager, top_k = 1)
res


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


[Document(metadata={'id': 'Attention_is_all_you_need_chunk_0002', 'chunk_index': 2, 'source_file': 'Attention is that all you need.pdf'}, page_content='1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15]. Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence len

In [10]:
import sys 
sys.path.append(r'D:\LLMOps\pyfiles')
from RetrievalSystem import RAGRetriever
        
rag_retrieve = RAGRetriever(vectore_store= transformers_table, embedding_manager= embedding_manager)

In [ ]:
question =  'Recurrent models typically factor computation along the symbol positions of'

res = rag_retrieve.retrieve(question, top_k=1, score_threshold=0.2)
res

Retrieveing documents for query:  'Recurrent models typically factor computation along the symbol positions of' 
Top k: 3, Score threshold: '0.2' 
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


Generated embeddings with shape: (1, 1024)
Retrieved 3 documents (after filtering)
RitrievalSytem loaded


[{'id': 'Attention_is_all_you_need_chunk_0002',
  'chunk_index': 2,
  'content': '1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15]. Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent wor

In [1]:
! uv add langchain

Resolved 139 packages in 73ms
Checked 112 packages in 517ms


In [ ]:

class AdvancedRAGPipeline:
    def __init__(self, retriever, model, processor, use_history: bool = True, history_window: int = 2):
        """
        use_history: system-level setting — if False, conversation history is never
                     tracked or included in prompts, regardless of what happens during queries.
        history_window: number of previous turns to include in the prompt when use_history is True.
        """
        self.retriever = retriever
        self.model = model
        self.processor = processor
        self.use_history = use_history
        self.history_window = history_window
        self.history = []

    def _build_inputs(self, prompt_text):
        """Builds model inputs using Gemma's chat template — required for multimodal processors."""
        messages = [{"role": "user", "content": prompt_text}]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(self.model.device)
        return inputs
   
#--------------------------------------------------  
    def generate(self, prompt_text, max_new_tokens=1024):
        """Generates a response all at once (no streaming) — used for summaries etc."""
        inputs = self._build_inputs(prompt_text)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=self.processor.tokenizer.eos_token_id
        )
        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        return self.processor.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def generate_streaming(self, prompt_text, max_new_tokens=512):
        """Generates a response token-by-token, streaming to console as it goes."""
        inputs = self._build_inputs(prompt_text)
        streamer = TextIteratorStreamer(self.processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=self.processor.tokenizer.eos_token_id
        )

        thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
        thread.start()

        full_response = ""
        for new_text in streamer:
            print(new_text, end="", flush=True)
            full_response += new_text

        thread.join()
        print()
        return full_response
#--------------------------------------------------      
#--------------------------------------------------  
    def query(self, question: str, top_k: int = 1, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> dict:
        results = self.retriever.retrieve(question, top_k=top_k)

        if not results:
            answer = "No relevant context found"
            sources = []
            context = ''

        else:
            context = '\n\n'.join([doc['content'] for doc in results])

            sources = [{
                'source': doc['source_file'],
                'rank': doc.get('rank'),
                'distances': doc.get('distance'),
                'chunk_index': doc['chunk_index'],
                'doc_id': doc['id'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

        # ----------------- build history + prompt -----------------
        if self.use_history:
            history_text = ""
            for turn in self.history[-self.history_window:]:
                history_text += f"Previous Question: {turn['question']}\nPrevious Answer: {turn['answer']}\n\n"

            prompt = f""" System_role: {'You are a enterprise ai assistant to a company, you reffer the user as Sir.'}

        Use the following context and conversation history to answer the question concisely.

        Conversation history: {history_text}
        Context: {context}
        Question: {question}
        """

        else:
            prompt = f""" System_role: {'You are a enterprise ai assistant to a company, you reffer the user as Sir.'}
        Use the following context to answer the question concisely.
         
        Context: {context}
        Question: {question}
        """
        # ------------------------------------------------------------

        if not results:
            answer = "No relevant context found"
        elif stream:
            print('Streaming answer:')
            answer = self.generate_streaming(prompt)
        else:
            answer = self.generate(prompt)

        citations = [f"[{i+1}] {src['source']}" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f'Summarize the following answer in 2 sentences: \n{answer}'
            summary = self.generate(summary_prompt)

        if self.use_history:
            self.history.append({
                'question': question,
                'answer': answer,
                'sources': sources,
                'summary': summary
            })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history if self.use_history else None
        }
#--------------------------------------------------      
#--------------------------------------------------       
query  = 'Recurrent models typically factor computation along the symbol positions of'

adv_rag = AdvancedRAGPipeline(rag_retrieve, model, processor= processor)  
result = adv_rag.query(query) 
  
print('\nFinal Answer:', result['answer'])
print('Summary:', result['summary'])
print('History:', result['history']) 